---
title: "Generate Cards for Group Member Listing"
jupyter: "card-lab"
execute: 
  enabled: true
format: 
    html: 
        default: false
---

In [1]:
#| output: false
#| eval: true
#| include: false
#| context: setup

# Load Excel sheet
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np

filename = "private/CARD Group Timeline.xlsx"

df = pd.read_excel(filename, sheet_name="People")
df.drop(0, inplace=True)
df.reset_index(inplace=True, drop=True)

names = df['Display Name'].values

# Get the current group members and alumni
finish_series = df['Ultimate Role Finish']
finish_text = finish_series.astype(str).str.strip().str.lower()
current_mask = finish_series.isna() | finish_text.isin(['', 'nan', 'nat']) | finish_text.str.contains('current', na=False)
alumni_mask = ~current_mask

current_group_member_indices = df[current_mask].index.tolist()
alumni_indices = df[alumni_mask].index.tolist()

current_group_members = df.loc[current_group_member_indices, 'Display Name']
alumni = df.loc[alumni_indices, 'Display Name']

current_index = np.zeros([len(names), 1])
current_index[current_group_member_indices] = 1
alumni_index = np.zeros([len(names), 1])
alumni_index[alumni_indices] = 1
df['current'] = current_index
df['alumni'] = alumni_index
del current_index, alumni_index

/Users/paytone/miniforge3/envs/card-lab/lib/python3.11/site-packages/openpyxl/worksheet/_read_only.py:85: UserWarning:

Data Validation extension is not supported and will be removed



In [2]:
#| output: false
#| eval: true
#| include: false
#| context: setup

# <!-- Future TODO: Store geocached data -->

import requests

def country_code_to_unicode_codepoints(country_code):
#def country_code_to_html_entities(country_code):
    """
    Convert an ISO 3166-1 alpha-2 country code to HTML character entities for flag emoji.
    
    Args:
        country_code (str): Two-letter country code, e.g., 'US', 'FR'
    
    Returns:
        str: HTML character entity string, e.g., '&#127482;&#127480;'
    """
    if not isinstance(country_code, str) or len(country_code) != 2 or not country_code.isalpha():
        raise ValueError("Input must be a 2-letter alphabetic ISO-3166-1 country code.")

    country_code = country_code.upper()
    entities = [
        f"&#{ord(c) - ord('A') + 0x1F1E6};"
        for c in country_code
    ]
    return ''.join(entities)

def get_wikipedia_url_from_country_code(iso_code):
    """
    Given an ISO 3166-1 alpha-2 country code, return the Wikipedia article URL for that country.

    Args:
        iso_code (str): Two-letter ISO country code (e.g., 'US', 'FR', 'JP').

    Returns:
        str: Wikipedia article URL, or None if code is invalid.
    """
    import pycountry
    import urllib.parse
    try:
        country = pycountry.countries.get(alpha_2=iso_code.upper())
        if country is None:
            return None
        country_name = country.name

        # Some countries have "official" long names that are more accurate for Wikipedia:
        # Try to use the "official_name" if available (e.g., for "Korea, Republic of")
        if hasattr(country, "official_name"):
            country_name = country.official_name

        # Encode for URL
        article_title = urllib.parse.quote(country_name.replace(" ", "_"))
        return f"https://en.wikipedia.org/wiki/{article_title}"

    except Exception as e:
        print(f"Error: {e}")
        return None

import urllib.parse

# Mapping from U.S. state abbreviation to full state name
US_STATE_NAMES = {
    "AL": "Alabama", "AK": "Alaska", "AZ": "Arizona", "AR": "Arkansas",
    "CA": "California", "CO": "Colorado", "CT": "Connecticut", "DE": "Delaware",
    "FL": "Florida", "GA": "Georgia", "HI": "Hawaii", "ID": "Idaho",
    "IL": "Illinois", "IN": "Indiana", "IA": "Iowa", "KS": "Kansas",
    "KY": "Kentucky", "LA": "Louisiana", "ME": "Maine", "MD": "Maryland",
    "MA": "Massachusetts", "MI": "Michigan", "MN": "Minnesota", "MS": "Mississippi",
    "MO": "Missouri", "MT": "Montana", "NE": "Nebraska", "NV": "Nevada",
    "NH": "New Hampshire", "NJ": "New Jersey", "NM": "New Mexico", "NY": "New York",
    "NC": "North Carolina", "ND": "North Dakota", "OH": "Ohio", "OK": "Oklahoma",
    "OR": "Oregon", "PA": "Pennsylvania", "RI": "Rhode Island", "SC": "South Carolina",
    "SD": "South Dakota", "TN": "Tennessee", "TX": "Texas", "UT": "Utah",
    "VT": "Vermont", "VA": "Virginia", "WA": "Washington", "WV": "West Virginia",
    "WI": "Wisconsin", "WY": "Wyoming",
    # Optionally add territories
    "DC": "District of Columbia", "PR": "Puerto Rico", "GU": "Guam", "VI": "United States Virgin Islands",
}

def get_us_state_wikipedia_url(abbreviation):
    """
    Given a two-letter U.S. state abbreviation, return the Wikipedia article URL.

    Args:
        abbreviation (str): e.g. 'CA', 'NY', 'TX'

    Returns:
        str: Wikipedia URL string, or None if abbreviation is invalid.
    """
    abbreviation = abbreviation.upper()
    state_name = US_STATE_NAMES.get(abbreviation)
    if not state_name:
        return None

    # Replace spaces with underscores and encode special characters
    article_title = urllib.parse.quote(state_name.replace(" ", "_"))
    return f"https://en.wikipedia.org/wiki/{article_title}"

def get_country_iso_code_nominatim(address, user_agent="country-lookup-script"):
    """
    Uses Nominatim (OpenStreetMap) to geocode an address and return the ISO 3166-1 alpha-2 country code.
    No API key required, but be respectful of usage limits.
    """
    url = "https://nominatim.openstreetmap.org/search"
    params = {
        'q': address,
        'format': 'json',
        'addressdetails': 1,
        'limit': 1,
    }
    headers = {
        'User-Agent': user_agent
    }

    try:
        response = requests.get(url, params=params, headers=headers)
        response.raise_for_status()
        data = response.json()

        if data and 'address' in data[0]:
            return data[0]['address'].get('country_code', '').upper()
        else:
            return None
    except Exception as e:
        print(f"Error during geocoding: {e}")
        return None

def get_country_flag_icon_url(country_code, format='svg'):
    """
    Returns the URL of the country flag icon from FlagCDN for a given ISO 3166-1 alpha-2 country code.
    
    Parameters:
        country_code (str): The two-letter country code, e.g., 'US', 'DE', 'FR'.
        format (str): 'svg' (default) or 'png'.
        
    Returns:
        str: URL to the flag image.
    """
    country_code = country_code.lower()
    if format == 'svg':
        return f"https://flagcdn.com/{country_code}.svg"
    elif format == 'png':
        return f"https://flagcdn.com/w80/{country_code}.png"
    else:
        raise ValueError("Format must be 'svg' or 'png'")

def get_us_state_flag_icon_url(us_state_code, format='svg'):
    """
    Returns the URL of the country flag icon from FlagCDN for a given ISO 3166-1 alpha-2 country code.
    
    Parameters:
        us_state_code (str): The two-letter country code, e.g., 'US', 'DE', 'FR'.
        format (str): 'svg' (default) or 'png'.
        
    Returns:
        str: URL to the flag image.
    """
    us_state_code = us_state_code.lower()
    if format == 'svg':
        return f"https://flagcdn.com/us-{us_state_code}.svg"
    elif format == 'png':
        return f"https://flagcdn.com/w80/us-{us_state_code}.png"
    else:
        raise ValueError("Format must be 'svg' or 'png'")

def get_us_state_code(address_string):
    import usaddress
    """
    Extracts the US state code (two-letter abbreviation) from an address string.

    Args:
        address_string (str): The full address string.

    Returns:
        str or None: The two-letter state code if found, otherwise None.
    """
    try:
        # Parse the address string
        parsed_address = usaddress.parse(address_string)

        # Iterate through the parsed components to find the state
        for component, label in parsed_address:
            if label == 'StateName':
                return component.upper()  # Return the state name in uppercase (e.g., 'CA')
        return None  # State not found
    except usaddress.RepeatedLabelError as e:
        print(f"Error parsing address: {e}")
        return None
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        return None

def get_zotero_items_by_author_and_type(author_name, allowed_types):
    from pyzotero import zotero
    zot = zotero.Zotero('5985739', 'group')

    search_initials = ''
    try:
        search_last, search_initials = author_name.split(",")
        search_last = search_last.strip()
        search_initials = search_initials.strip()
    except:
        search_last = author_name

    items = zot.items(q=search_last, qmode='everything')

    if len(search_initials) > 0:
        filtered_items = []
        for item in items:
            data = item['data']
            item_type = data.get('itemType', '')
            creators = data.get('creators', [])

            if item_type in allowed_types:
                if matches_author(creators, search_last, search_initials):
                    filtered_items.append(item)
    else:
        filtered_items = []
        for item in items:
            data = item['data']
            item_type = data.get('itemType', '')
            creators = data.get('creators', [])

            if item_type in allowed_types:
                if any(author_name.lower() in c.get('lastName', '').lower() for c in creators):
                    filtered_items.append(item)

    return filtered_items

def get_formatted_citations(item_keys, style):
    import requests
    from bs4 import BeautifulSoup

    if not item_keys:
        return ''

    keys_csv = ",".join(item_keys)
    url = f"https://api.zotero.org/groups/5985739/items"
    headers = {
        'Accept': 'text/html'  # Or text/plain if preferred
    }
    params = {
        'itemKey': keys_csv,
        'format': 'bib',
        'style': style,
        'sort': 'date',
        'direction': 'desc'
    }

    response = requests.get(url, headers=headers, params=params)
    if response.status_code != 200:
        fallback_params = {
            'itemKey': keys_csv,
            'format': 'bib',
            'style': style
        }
        response = requests.get(url, headers=headers, params=fallback_params)

    if response.status_code != 200:
        return ''

    import re

    def _entry_sort_key(entry_text):
        month_map = {
            'jan': 1, 'feb': 2, 'mar': 3, 'apr': 4, 'may': 5, 'jun': 6,
            'jul': 7, 'aug': 8, 'sep': 9, 'oct': 10, 'nov': 11, 'dec': 12
        }

        iso_match = re.search(r'(19|20)\d{2}-(\d{2})-(\d{2})', entry_text)
        if iso_match:
            year = int(iso_match.group(0)[0:4])
            month = int(iso_match.group(2))
            day = int(iso_match.group(3))
            return (year, month, day)

        month_year_match = re.search(r'(Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)\.?\s+((?:19|20)\d{2})', entry_text, re.IGNORECASE)
        if month_year_match:
            month = month_map[month_year_match.group(1).lower()]
            year = int(month_year_match.group(2))
            return (year, month, 0)

        year_match = re.search(r'(19|20)\d{2}', entry_text)
        if year_match:
            return (int(year_match.group(0)), 0, 0)

        return (0, 0, 0)

    soup = BeautifulSoup(response.text, 'html.parser')
    bib_body = soup.select_one('div.csl-bib-body')
    if bib_body is None:
        return response.text

    entries = bib_body.find_all('div', class_='csl-entry', recursive=False)
    total_entries = len(entries)

    if total_entries == 0:
        return response.text

    ordered_entries = sorted(
        entries,
        key=lambda entry: _entry_sort_key(entry.get_text(' ', strip=True)),
        reverse=True
    )

    for entry in entries:
        entry.extract()

    for index, entry in enumerate(ordered_entries):
        display_index = total_entries - index
        left_margin = entry.select_one('div.csl-left-margin')
        if left_margin:
            left_margin.clear()
            left_margin.append(f"[{display_index}]")
        bib_body.append(entry)

    rendered = str(soup)
    rendered = re.sub(r'^<\?xml[^>]*>\s*', '', rendered)
    return rendered

def _creator_display_name(creator):
    if creator.get('name'):
        return creator.get('name')

    first = creator.get('firstName', '').strip()
    last = creator.get('lastName', '').strip()
    first_parts = [part for part in first.replace('-', ' ').split() if part]
    initials = ' '.join([f"{part[0].upper()}." for part in first_parts])

    if initials and last:
        return f"{initials} {last}"
    if last:
        return last
    if initials:
        return initials

    full = f"{first} {last}".strip()
    return full

def get_presentation_citations_all_authors(items):
    import html
    import re
    from datetime import datetime

    def _presentation_sort_key(raw_date):
        text = str(raw_date or '').strip()
        if not text:
            return (0, 0, 0)

        formats = [
            '%Y-%m-%d', '%Y-%m', '%Y',
            '%B %d, %Y', '%b %d, %Y',
            '%B %Y', '%b %Y',
        ]

        for fmt in formats:
            try:
                parsed = datetime.strptime(text, fmt)
                return (parsed.year, parsed.month, parsed.day)
            except ValueError:
                continue

        year_match = re.search(r'(19|20)\d{2}', text)
        if year_match:
            return (int(year_match.group(0)), 0, 0)

        return (0, 0, 0)

    entries = []
    for item in items:
        data = item.get('data', {})
        ordered_creators = []

        for creator in data.get('creators', []):
            display_name = _creator_display_name(creator)
            if not display_name:
                continue

            safe_name = html.escape(display_name)
            creator_type = str(creator.get('creatorType', '')).lower()
            if creator_type == 'presenter':
                ordered_creators.append(f"<em>{safe_name}</em> (presenter)")
            else:
                ordered_creators.append(safe_name)

        authors_text = ', '.join(ordered_creators) if ordered_creators else 'Unknown author'

        raw_date = data.get('date', 'n.d.')
        date = html.escape(str(raw_date))
        title = html.escape(str(data.get('title', 'Untitled')))

        meeting_name = data.get('meetingName') or data.get('proceedingsTitle') or ''
        meeting_name = html.escape(str(meeting_name))

        if meeting_name:
            entry = f"{authors_text}. ({date}). {title}. <em>{meeting_name}</em>."
        else:
            entry = f"{authors_text}. ({date}). {title}."

        entries.append((_presentation_sort_key(raw_date), entry))

    if not entries:
        return ''

    entries.sort(key=lambda pair: pair[0], reverse=True)
    rendered = []
    total_entries = len(entries)
    for index, (_, entry_text) in enumerate(entries):
        display_index = total_entries - index
        rendered.append(
            '  <div class="csl-entry" style="clear: left; ">\n'
            f'    <div class="csl-left-margin" style="float: left; padding-right: 0.5em; text-align: right; width: 2em;">[{display_index}]</div>'
            f'<div class="csl-right-inline" style="margin: 0 .4em 0 2.5em;">{entry_text}</div>\n'
            '  </div>'
        )

    return '<div class="csl-bib-body" style="line-height: 1.35; ">\n' + '\n'.join(rendered) + '\n</div>'

def get_people_workproducts_assets():
    return """```{=html}
<style>
.citation-badges { display: inline-flex; align-items: center; gap: 6px; margin-left: 6px; }
.oa-icon { margin-left: 6px; vertical-align: middle; display: inline-flex; align-items: center; }
.oa-gold { color: #FFD700; } .oa-green { color: #2e7d32; } .oa-bronze { color: #cd7f32; }
.oa-hybrid { color: #ff8f00; } .oa-diamond { color: #b9f2ff; } .oa-unknown { color: #1e88e5; }
.pdf-icon { margin-left: 4px; color: #e00122; vertical-align: middle; display: inline-flex; align-items: center; }
</style>
<script type='text/javascript' src='//d1bxh8uas1mnw7.cloudfront.net/assets/embed.js'></script>
<script async src='https://badge.dimensions.ai/badge.js' charset='utf-8'></script>
<script src='https://code.iconify.design/2/2.2.1/iconify.min.js'></script>
<script>(function(){function c(s){switch(s){case 'gold':return 'oa-gold';case 'green':return 'oa-green';case 'bronze':return 'oa-bronze';case 'hybrid':return 'oa-hybrid';case 'diamond':return 'oa-diamond';default:return 'oa-unknown';}}function d(t){if(!t)return null;const m=t.match(/10\.\d{4,9}\/[-._;()/:A-Z0-9]+/i);return m?m[0].toLowerCase().replace(/[.,;]$/,''):null;}async function e(){const n=[...document.querySelectorAll('.csl-entry,.citation')];const map=new Map();const dois=[];n.forEach((node)=>{let doi=null;for(const a of node.querySelectorAll('a[href]')){const h=a.getAttribute('href')||'';const k=h.includes('doi.org/')?h.split('doi.org/').pop():null;doi=d(k||a.textContent||h);if(doi)break;}if(!doi)doi=d(node.textContent||'');if(doi){map.set(doi,node);dois.push(doi);}});if(!dois.length)return;const chunks=[];for(let i=0;i<dois.length;i+=100)chunks.push(dois.slice(i,i+100));const works=[];for(const p of chunks){const f=p.map((x)=>encodeURIComponent(x)).join('|');try{const r=await fetch(`https://api.openalex.org/works?filter=doi:${f}`);if(!r.ok)continue;const j=await r.json();if(Array.isArray(j.results))works.push(...j.results);}catch(_){}}works.forEach((w)=>{const doi=(w.doi||'').replace(/^https?:\/\/doi.org\//i,'').toLowerCase();const node=map.get(doi);if(!node||node.querySelector('[data-people-citation-badges]'))return;const b=document.createElement('span');b.className='citation-badges';b.setAttribute('data-people-citation-badges','1');const a=document.createElement('span');a.className='altmetric-embed';a.setAttribute('data-badge-type','4');a.setAttribute('data-doi',doi);a.setAttribute('data-hide-no-mentions','true');b.appendChild(a);const dm=document.createElement('span');dm.className='__dimensions_badge_embed__';dm.setAttribute('data-style','small_rectangle');dm.setAttribute('data-doi',doi);dm.setAttribute('data-hide-zero-citations','true');b.appendChild(dm);if(w.open_access&&w.open_access.is_oa&&w.open_access.oa_url){const ol=document.createElement('a');ol.href=w.open_access.oa_url;ol.target='_blank';ol.className='oa-icon-link oa-icon '+c(w.open_access.oa_status||'unknown');const oi=document.createElement('span');oi.className='iconify';oi.setAttribute('data-icon','academicons:open-access');oi.setAttribute('data-width','20');oi.setAttribute('data-height','20');ol.appendChild(oi);b.appendChild(ol);const arr=w.open_access.oa_urls||[];const rx=/\.pdf(\?|$)/i;let pdf=null;if(Array.isArray(arr)){const pe=arr.find((x)=>rx.test((x&&x.url)||''));if(pe)pdf=pe.url;}if(!pdf&&rx.test(w.open_access.oa_url))pdf=w.open_access.oa_url;if(pdf){const pl=document.createElement('a');pl.href=pdf;pl.target='_blank';pl.className='pdf-icon-link pdf-icon';const pi=document.createElement('span');pi.className='iconify';pi.setAttribute('data-icon','mdi:file-pdf-box');pi.setAttribute('data-width','20');pi.setAttribute('data-height','20');pl.appendChild(pi);b.appendChild(pl);}}node.appendChild(b);});if(typeof _altmetric_embed_init==='function')_altmetric_embed_init();if(window.__dimensions_embed&&window.__dimensions_embed.addBadges)window.__dimensions_embed.addBadges();if(window.Iconify&&window.Iconify.scan)window.Iconify.scan();}if(document.readyState==='loading'){document.addEventListener('DOMContentLoaded',e);}else{e();}})();</script>
```"""

def matches_author(creators, search_last, search_initials):
    """
    Check if any creator matches the given last name and initials.
    """
    for creator in creators:
        last = creator.get('lastName', '').lower()
        first = creator.get('firstName', '').lower()
        if not first:
            continue
        
        first_initials = ''.join([part[0] for part in first.split() if part])
        
        if last == search_last.lower() and first_initials.upper().startswith(search_initials.upper()):
            return True
    return False

def get_role_categories(row):
    role_columns = [
        'Ultimate Degree-Role',
        'Penultimate Degree-Role',
        'Antepenultimate Degree-Role',
        'Preantepenultimate Degree-Role',
        'Propreantepenultimate Degree-Role',
        'Ultrasuprapropreantepenultimate Degree-Role',
    ]

    categories = []
    seen = set()
    for col in role_columns:
        value = row.get(col, None)
        if pd.isnull(value):
            continue

        text = str(value).strip()
        if not text or text.lower() in {'nan', 'nat', 'none'}:
            continue

        if text not in seen:
            categories.append(text)
            seen.add(text)

    if not categories:
        categories.append('Uncategorized')

    return categories

In [3]:
#| output: false
#| eval: true
#| include: false
#| context: setup

# Generate cards for current group members
import os
import shutil

citation_format = "ieee"

def save_with_dirs(path, array, **kwargs):
    """
    Save a NumPy array to a file, creating directories if needed.

    Parameters
    ----------
    path : str
        Full path (including filename) where the array will be saved.
    array : np.ndarray
        The data to save.
    **kwargs
        Additional arguments passed to numpy.savetxt.
    """
    # Get directory part of the path
    dir_path = os.path.dirname(path)

    if dir_path:  # Only try to create if there's a directory specified
        os.makedirs(dir_path, exist_ok=True)

    # Save the array
    np.savetxt(path, array, **kwargs)

for index, row in df[df['current']==True].iterrows():

   linkedin = ''
   orcid = ''
   github =''
   googlescholar = ''
   current_role = row.get('Ultimate Degree-Role', None)
   if pd.isnull(current_role) or not str(current_role).strip() or str(current_role).strip().lower() in {'nan', 'nat', 'none'}:
      categories = ['Uncategorized']
   else:
      categories = [str(current_role).strip()]
   categories_yaml = ', '.join([f'"{item}"' for item in categories])
   txt = "".join(['---\n'+\
               'title: "{0}"\n'+\
               'categories: [{1}]\n'+\
               'member-since: {2}\n'+\
               'date: today\n'+\
               'date-modified: {2}\n'+\
               'date-format: "MMMM YYYY"\n'+\
               'language:\n'+\
               '  title-block-published: "Updated"\n'+\
               '  title-block-modified: "Joined"\n'+\
               'execute:\n' +\
               '  echo: false\n' +\
               'image: ']).format(row['Display Name'],
                                  categories_yaml, 
                                  row["Ultimate Role Start"])
   
   member_slug = row['Display Name'].replace(' ', '_')
   private_headshot_path = "private/Group Member Photos/{0}.jpg".format(member_slug)
   public_headshot_path = "files/photos/People/{0}.jpg".format(member_slug)
   headshot = "../../{0}".format(public_headshot_path)
   placeholder = "../../files/photos/People/{0}.jpg".format('anon')
   if os.path.exists(private_headshot_path):
      os.makedirs("files/photos/People", exist_ok=True)
      shutil.copy2(private_headshot_path, public_headshot_path)
      txt += "{0}\n".format(headshot)
      txt += "".join(['about:\n'+\
               '  template: trestles\n'+\
               '  image: {0}\n',
               '  image-alt: "Photo of {1}"\n']).format(headshot, row['Display Name'])
   elif os.path.exists(public_headshot_path):
      txt += "{0}\n".format(headshot)
      txt += "".join(['about:\n'+\
               '  template: trestles\n'+\
               '  image: {0}\n',
               '  image-alt: "Photo of {1}"\n']).format(headshot, row['Display Name'])
   else:
      txt += "{0}\n".format(placeholder)
      txt += "".join(['about:\n'+\
               '  template: trestles\n'+\
               '  image: {0}\n',
               '  image-alt: "Stock photo of a dog wearing glasses."\n']).format(placeholder)

   txt += "  image-shape: round\n"
   links = []

   if not pd.isnull(row['LinkedIn']):
      links.append(
         '    - text: "{{{{< iconify mdi linkedin >}}}}"\n'
         '      url: https://www.linkedin.com/in/{0}\n'.format(row['LinkedIn'])
      )
   if not pd.isnull(row['ORCID']):
      links.append(
         '    - text: "{{{{< iconify simple-icons orcid >}}}}"\n'
         '      url: https://orcid.org/{0}\n'.format(row['ORCID'])
      )
   if not pd.isnull(row['Google Scholar']):
      links.append(
         '    - text: "{{{{< iconify academicons google-scholar >}}}}"\n'
         '      url: https://scholar.google.com/citations?user={0}&hl=en\n'.format(row['Google Scholar'])
      )
   if not pd.isnull(row['GitHub']):
      links.append(
         '    - text: "{{{{< iconify mdi github >}}}}"\n'
         '      url: https://github.com/{0}\n'.format(row['GitHub'])
      )

   state_codes = []
   country_codes = []
   for i in range(1,10):
      col_name = "Hometown {0}".format(i)
      if not pd.isnull(row[col_name]):
         code = get_country_iso_code_nominatim(row[col_name])
         country_codes.append(code)
         
         if code == "US":
               state_codes.append(get_us_state_code(row[col_name]))
    
   # Store only unique values
   country_codes = list(set(
      [item for item in country_codes if item is not None]))
   state_codes = list(set(
      [item for item in state_codes if item is not None]))

   flags = []
   for item in country_codes:
      url = get_country_flag_icon_url(item, 'svg')
      links.append(
         '    - text: "![]({0}){{width=0.25in}}"\n'
         '      url: {1}\n'.format(url, get_wikipedia_url_from_country_code(item))
      )
   for item in state_codes:
      url = get_us_state_flag_icon_url(item, 'svg')
      links.append(
         '    - text: "![]({0}){{width=0.25in}}"\n'
         '      url: {1}\n'.format(url, get_us_state_wikipedia_url(item))
      )

   if links:
      txt += "  links:\n"
      txt += ''.join(links)

   txt += '\n---\n\n'

   people_workproducts_assets = get_people_workproducts_assets()
   publications = ''

   if pd.isnull(row['Authorship Name']):
      nom = row['Display Name'].split(' ')[-1]
   else:
      nom = row['Authorship Name'].split('.')[0:1][0]

   results = get_zotero_items_by_author_and_type(
      nom, ['thesis'])
   item_keys = [item['key'] for item in results]
   if len(item_keys) > 0:
      publications += '# Thesis/Dissertation\n\n' + \
      "```{=html}\n" +\
      get_formatted_citations(item_keys, citation_format) + '```\n\n'

   results = get_zotero_items_by_author_and_type(
      nom, ['journalArticle', 'book', 'bookSection', 'conferencePaper', 'report'])
   item_keys = [item['key'] for item in results]
   if len(item_keys) > 0:
      publications += '# Publications\n\n' + \
      "```{=html}\n" +\
      get_formatted_citations(item_keys, citation_format) + '```\n\n'

   results = get_zotero_items_by_author_and_type(
      nom, ['presentation'])
   item_keys = [item['key'] for item in results]
   if len(item_keys) > 0:
      presentation_html = get_presentation_citations_all_authors(results)
      publications += '# Presentations\n\n' + \
      "```{=html}\n" +\
      presentation_html + '\n```\n\n'

   results = get_zotero_items_by_author_and_type(
      nom, ['computerProgram'])
   item_keys = [item['key'] for item in results]
   if len(item_keys) > 0:
      publications += '# Code & Datasets\n\n' + \
      "```{=html}\n" +\
      get_formatted_citations(item_keys, citation_format) + '```\n\n'
   
   publication = ''.join(['*CARD Lab Work Products:*\n', publications])

   x = np.array(['\n'.join([txt, people_workproducts_assets, publications])])
   filepath = ''.join(['./people/current/', row['Display Name'].replace(' ', '_'), '.qmd'])
   save_with_dirs(filepath, x, fmt='%s')


/var/folders/nz/l5ynytfs6sj2drjd5cs1wckc0000gn/T/ipykernel_82333/145420493.py:287: XMLParsedAsHTMLWarning:

It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)




In [4]:
#| output: false
#| eval: true
#| include: false
#| context: setup

# Python cell to generate a listing .qmd file
listing_qmd_path = "people/current.qmd"

listing_yaml = """
---
title: "Current Group Members"
format: html
listing:
  type: grid
  contents: current
  template: _partials/people-listing.ejs.md
  sort: "date asc"
  image-placeholder: files/images/anon.jpg
  date-format: "MMM YYYY"
  categories: numbered
  image-height: 225px
  grid-columns: 4
  max-items: 100 # Show all items
  filter-ui: [categories, date, title]
include-in-header:
  text: |
      <style>
      /* Dashboard grid listing tweaks! /*

      /* Hide pop-out / maximize controls for listing items */
      .quarto-listing .listing-item .card-header-actions {
      display: none !important;
      }

      .quarto-listing .listing-item .card-header {
      cursor: default;
      }
      </style>
---
"""

# Write the YAML to the .qmd file
with open(listing_qmd_path, "w") as f:
    f.write(listing_yaml)

print(f"Listing page written to {listing_qmd_path}")

Listing page written to people/current.qmd


In [5]:
#| output: false
#| eval: true
#| include: false
#| context: setup

# Generate cards for group alumni
import os
import shutil

citation_format = "ieee"

def save_with_dirs(path, array, **kwargs):
    """
    Save a NumPy array to a file, creating directories if needed.

    Parameters
    ----------
    path : str
        Full path (including filename) where the array will be saved.
    array : np.ndarray
        The data to save.
    **kwargs
        Additional arguments passed to numpy.savetxt.
    """
    # Get directory part of the path
    dir_path = os.path.dirname(path)

    if dir_path:  # Only try to create if there's a directory specified
        os.makedirs(dir_path, exist_ok=True)

    # Save the array
    np.savetxt(path, array, **kwargs)

def format_member_date(value):
    if pd.isnull(value):
        return ''
    return pd.to_datetime(value).strftime('%B %Y')

for index, row in df[df['current']==False].iterrows():

   linkedin = ''
   orcid = ''
   github =''
   googlescholar = ''
   categories = get_role_categories(row)
   categories_yaml = ', '.join([f'"{item}"' for item in categories])
   member_range = '{0} to {1}'.format(format_member_date(row["Ultimate Role Start"]), format_member_date(row["Ultimate Role Finish"]))
   txt = "".join(['---\n'+\
               'title: "{0}"\n'+\
               'categories: [{1}]\n'+\
               'member-from: {2}\n'+\
               'member-to: {3}\n'+\
               'date: today\n'+\
               'date-modified: {2}\n'+\
               'date-format: "MMM YYYY"\n'+\
               'language:\n'+\
               '  title-block-published: "Updated"\n'+\
               '  title-block-modified: "Member from"\n'+\
               'execute:\n' +\
               '  echo: false\n' +\
               'image: ']).format(row['Display Name'],
                                  categories_yaml, 
                                  row["Ultimate Role Start"],
                                  row["Ultimate Role Finish"])
   
   member_slug = row['Display Name'].replace(' ', '_')
   private_headshot_path = "private/Group Member Photos/{0}.jpg".format(member_slug)
   public_headshot_path = "files/photos/People/{0}.jpg".format(member_slug)
   headshot = "../../{0}".format(public_headshot_path)
   placeholder = "../../files/photos/People/{0}.jpg".format('anon')
   if os.path.exists(private_headshot_path):
      os.makedirs("files/photos/People", exist_ok=True)
      shutil.copy2(private_headshot_path, public_headshot_path)
      txt += "{0}\n".format(headshot)
      txt += "".join(['about:\n'+\
               '  template: trestles\n'+\
               '  image: {0}\n',
               '  image-alt: "Photo of {1}"\n']).format(headshot, row['Display Name'])
   elif os.path.exists(public_headshot_path):
      txt += "{0}\n".format(headshot)
      txt += "".join(['about:\n'+\
               '  template: trestles\n'+\
               '  image: {0}\n',
               '  image-alt: "Photo of {1}"\n']).format(headshot, row['Display Name'])
   else:
      txt += "{0}\n".format(placeholder)
      txt += "".join(['about:\n'+\
               '  template: trestles\n'+\
               '  image: {0}\n',
               '  image-alt: "Stock photo of a dog wearing glasses."\n']).format(placeholder)

   txt += "  image-shape: round\n"
   links = []

   if not pd.isnull(row['LinkedIn']):
      links.append(
         '    - text: "{{{{< iconify mdi linkedin >}}}}"\n'
         '      url: https://www.linkedin.com/in/{0}\n'.format(row['LinkedIn'])
      )
   if not pd.isnull(row['ORCID']):
      links.append(
         '    - text: "{{{{< iconify simple-icons orcid >}}}}"\n'
         '      url: https://orcid.org/{0}\n'.format(row['ORCID'])
      )
   if not pd.isnull(row['Google Scholar']):
      links.append(
         '    - text: "{{{{< iconify academicons google-scholar >}}}}"\n'
         '      url: https://scholar.google.com/citations?user={0}&hl=en\n'.format(row['Google Scholar'])
      )
   if not pd.isnull(row['GitHub']):
      links.append(
         '    - text: "{{{{< iconify mdi github >}}}}"\n'
         '      url: https://github.com/{0}\n'.format(row['GitHub'])
      )

   state_codes = []
   country_codes = []
   for i in range(1,10):
      col_name = "Hometown {0}".format(i)
      if not pd.isnull(row[col_name]):
         code = get_country_iso_code_nominatim(row[col_name])
         country_codes.append(code)
         
         if code == "US":
               state_codes.append(get_us_state_code(row[col_name]))
    
   # Store only unique values
   country_codes = list(set(
      [item for item in country_codes if item is not None]))
   state_codes = list(set(
      [item for item in state_codes if item is not None]))

   flags = []
   for item in country_codes:
      url = get_country_flag_icon_url(item, 'svg')
      links.append(
         '    - text: "![]({0}){{width=0.25in}}"\n'
         '      url: {1}\n'.format(url, get_wikipedia_url_from_country_code(item))
      )
   for item in state_codes:
      url = get_us_state_flag_icon_url(item, 'svg')
      links.append(
         '    - text: "![]({0}){{width=0.25in}}"\n'
         '      url: {1}\n'.format(url, get_us_state_wikipedia_url(item))
      )

   if links:
      txt += "  links:\n"
      txt += ''.join(links)

   txt += '\n---\n\n'
   txt += ''.join([
      '```{=html}\n',
      '<div class="member-range-source" data-member-range="' + member_range + '"></div>\n',
      '<script>\n',
      'document.addEventListener("DOMContentLoaded", function () {\n',
      '  const source = document.querySelector(".member-range-source");\n',
      '  const modified = document.querySelector(".quarto-title-meta .date-modified");\n',
      '  if (source && modified) {\n',
      '    const range = source.dataset.memberRange || modified.textContent;\n',
      '    const contents = modified.closest(".quarto-title-meta-contents");\n',
      '    const heading = contents ? contents.previousElementSibling : null;\n',
      '    if (heading && heading.classList.contains("quarto-title-meta-heading")) {\n',
      '      heading.textContent = `Member from ${range}`;\n',
      '      if (contents) { contents.remove(); }\n',
      '    } else {\n',
      '      modified.textContent = range;\n',
      '    }\n',
      '  }\n',
      '});\n',
      '</script>\n',
      '```\n\n'])

   people_workproducts_assets = get_people_workproducts_assets()
   publications = ''

   if pd.isnull(row['Authorship Name']):
      nom = row['Display Name'].split(' ')[-1]
   else:
      nom = row['Authorship Name'].split('.')[0:1][0]

   results = get_zotero_items_by_author_and_type(
      nom, ['thesis'])
   item_keys = [item['key'] for item in results]
   if len(item_keys) > 0:
      publications += '# Thesis/Dissertation\n\n' + \
      "```{=html}\n" +\
      get_formatted_citations(item_keys, citation_format) + '```\n\n'

   results = get_zotero_items_by_author_and_type(
      nom, ['journalArticle', 'book', 'bookSection', 'conferencePaper', 'report'])
   item_keys = [item['key'] for item in results]
   if len(item_keys) > 0:
      publications += '# Publications\n\n' + \
      "```{=html}\n" +\
      get_formatted_citations(item_keys, citation_format) + '```\n\n'

   results = get_zotero_items_by_author_and_type(
      nom, ['presentation'])
   item_keys = [item['key'] for item in results]
   if len(item_keys) > 0:
      presentation_html = get_presentation_citations_all_authors(results)
      publications += '# Presentations\n\n' + \
      "```{=html}\n" +\
      presentation_html + '\n```\n\n'

   results = get_zotero_items_by_author_and_type(
      nom, ['computerProgram'])
   item_keys = [item['key'] for item in results]
   if len(item_keys) > 0:
      publications += '# Code & Datasets\n\n' + \
      "```{=html}\n" +\
      get_formatted_citations(item_keys, citation_format) + '```\n\n'
   
   publication = ''.join(['*CARD Lab Work Products:*\n', publications])

   x = np.array(['\n'.join([txt, people_workproducts_assets, publications])])
   filepath = ''.join(['./people/alumni/', row['Display Name'].replace(' ', '_'), '.qmd'])
   save_with_dirs(filepath, x, fmt='%s')

/var/folders/nz/l5ynytfs6sj2drjd5cs1wckc0000gn/T/ipykernel_82333/145420493.py:287: XMLParsedAsHTMLWarning:

It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)




In [6]:
#| output: false
#| eval: true
#| include: false
#| context: setup

# Python cell to generate a listing .qmd file
listing_qmd_path = "people/alumni.qmd"

listing_yaml = """
---
title: "Alumni"
format: html
listing:
  type: grid
  contents: alumni
  template: _partials/people-listing.ejs.md
  sort: "member-to desc"
  image-placeholder: files/images/anon.jpg
  date-format: "MMM YYYY"
  image-height: 225px
  grid-columns: 4
  page-size: 100
  categories: numbered
  filter-ui: [categories, date, title]
include-in-header:
  text: |
      <style>
      /* Hide pop-out / maximize controls for listing items */
      .quarto-listing .listing-item .card-header-actions {
      display: none !important;
      }

      .quarto-listing .listing-item .card-header {
      cursor: default;
      }
      </style>
---
"""

# Write the YAML to the .qmd file
with open(listing_qmd_path, "w") as f:
    f.write(listing_yaml)

print(f"Listing page written to {listing_qmd_path}")

Listing page written to people/alumni.qmd


In [7]:
#| output: true
#| eval: true
#| include: false
#| context: setup

import subprocess, sys
result = subprocess.run(
    [sys.executable, "scripts/recycle-stale-people.py"],
    capture_output=True, text=True
)
if result.stdout:
    print(result.stdout)
if result.returncode != 0 and result.stderr:
    print(result.stderr, file=sys.stderr)


[recycle-stale-people] Checking for stale profile files...
  No stale current profiles found.
  No stale alumni profiles found.
[recycle-stale-people] Done. Total recycled: 0

